# Building a baseline model 

In [20]:
import numpy as np
import pandas as pd

from restaurant_visitor_eda.config import PROCESSED_DATA_DIR

df = pd.read_csv(PROCESSED_DATA_DIR / "train_baseline.csv")
df["visit_date"] = pd.to_datetime(df["visit_date"])
df = df.sort_values("visit_date").reset_index(drop=True)

In [21]:
X = df[
    [
        "air_store_id",
        "air_genre_name",
        "latitude",
        "longitude",
        "day_of_week",
        "month",
        "day_pattern",
        "prefecture",
        "district",
        "block",
    ]
]

y = np.log1p(df["visitors"])

In [22]:
numeric = ["latitude", "longitude"]

categorical = [
    "air_store_id",
    "air_genre_name",
    "day_of_week",
    "month",
    "day_pattern",
    "prefecture",
    "district",
    "block",
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

column_transformer = ColumnTransformer(
    [
        ("ohe", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("scaling", StandardScaler(), numeric),
    ]
)

pipeline = Pipeline(
    steps=[("ohe_and_scaling", column_transformer), ("regression", Ridge(random_state=42))]
)


tscv = TimeSeriesSplit(n_splits=15)

param_grid = {"regression__alpha": np.logspace(-1, 2)}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    # verbose=2
)

grid_search.fit(X, y)

print(f"Best alpha: {grid_search.best_params_}")
best_score = -grid_search.best_score_
print(f"Best RMSLE: {best_score:.4f}")

Best alpha: {'regression__alpha': np.float64(3.906939937054617)}
Best RMSLE: 0.5971


In [24]:
from restaurant_visitor_eda.plots import plot_cv_error_vs_alpha

plot_cv_error_vs_alpha(grid_search.cv_results_)